
# DATA 715 Group Breakout
## OMOP Feature Export Analysis in Google Colab

This notebook continues the SQL breakout exercise.

Each group should export its patient-level feature table from MySQL as a CSV file containing:

- `person_id`
- `study_arm`
- `last_drug_date`
- `f1`
- `f2`
- `y`

Where:

- `f1` = number of drug exposures in the 365 days ending on the patient's index date
- `f2` = number of distinct condition concepts represented after hierarchy expansion
- `y` = 1 if the patient died within 30 days after the index date, otherwise 0

The goal is to move from a relational database feature-engineering workflow into Python for characterization, visualization, and modeling.


## 1. Import Python Libraries

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")



## 2. Upload Your SQL Feature Export

Run the next cell and select the CSV file exported from MySQL.

Suggested filenames include:

- `CONTROL_features.csv`
- `INTERVENTION_A_features.csv`
- `INTERVENTION_B_features.csv`


In [ ]:

from google.colab import files

uploaded = files.upload()

if len(uploaded) == 0:
    raise ValueError("No file was uploaded.")

filename = next(iter(uploaded))
print(f"Using file: {filename}")


## 3. Load and Validate the Dataset

In [ ]:

df = pd.read_csv(filename)

expected_columns = {
    "person_id",
    "study_arm",
    "last_drug_date",
    "f1",
    "f2",
    "y"
}

missing_columns = expected_columns - set(df.columns)

if missing_columns:
    raise ValueError(
        f"The file is missing required columns: {sorted(missing_columns)}"
    )

df["last_drug_date"] = pd.to_datetime(
    df["last_drug_date"],
    errors="coerce"
)

print(f"Rows: {len(df):,}")
print(f"Distinct patients: {df['person_id'].nunique():,}")
print(f"Study arms: {df['study_arm'].nunique():,}")

df.head()


## 4. Check One Row Per Patient

In [ ]:

row_count = len(df)
patient_count = df["person_id"].nunique()

print(f"Rows: {row_count:,}")
print(f"Distinct patients: {patient_count:,}")

if row_count == patient_count:
    print("PASS: one row per patient")
else:
    print("WARNING: duplicate patient rows exist")

duplicates = df[df.duplicated("person_id", keep=False)]

if len(duplicates) > 0:
    display(
        duplicates.sort_values("person_id").head(20)
    )


## 5. Missing Data Characterization

In [ ]:

missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": df.isna().mean() * 100
}).sort_values(
    "missing_percent",
    ascending=False
)

display(missing_summary)


## 6. Study Arm Summary

In [ ]:

study_arm_summary = (
    df.groupby("study_arm")
      .agg(
          patients=("person_id", "nunique"),
          mean_f1=("f1", "mean"),
          median_f1=("f1", "median"),
          mean_f2=("f2", "mean"),
          median_f2=("f2", "median"),
          deaths=("y", "sum"),
          mortality_rate=("y", "mean")
      )
      .reset_index()
)

study_arm_summary["mortality_percent"] = (
    study_arm_summary["mortality_rate"] * 100
)

display(study_arm_summary)


## 7. Feature Characterization

In [ ]:

feature_summary = df[["f1", "f2", "y"]].describe().T

feature_summary["median"] = df[["f1", "f2", "y"]].median()

display(feature_summary)


## 8. Outcome Distribution

In [ ]:

outcome_summary = (
    df["y"]
      .value_counts(dropna=False)
      .rename_axis("y")
      .reset_index(name="patients")
)

outcome_summary["percent"] = (
    outcome_summary["patients"] / len(df) * 100
)

display(outcome_summary)


In [ ]:

outcome_counts = df["y"].value_counts().sort_index()

plt.figure(figsize=(7, 5))
plt.bar(
    outcome_counts.index.astype(str),
    outcome_counts.values
)
plt.xlabel("30-Day Mortality Outcome")
plt.ylabel("Patients")
plt.title("Outcome Distribution")
plt.show()


## 9. Visualize the Engineered Features

In [ ]:

plt.figure(figsize=(8, 6))

scatter = plt.scatter(
    df["f1"],
    df["f2"],
    c=df["y"],
    alpha=0.6
)

plt.xlabel("f1: Drug Exposures in Previous 365 Days")
plt.ylabel("f2: Condition Hierarchy Feature")
plt.title("Patient Feature Space")
plt.colorbar(scatter, label="30-Day Mortality Outcome")
plt.show()


In [ ]:

plt.figure(figsize=(8, 5))

groups = [
    group["f1"].dropna().values
    for _, group in df.groupby("y")
]

labels = [
    str(label)
    for label in sorted(df["y"].dropna().unique())
]

plt.boxplot(
    groups,
    tick_labels=labels
)

plt.xlabel("30-Day Mortality Outcome")
plt.ylabel("f1: Drug Exposures")
plt.title("Drug Exposure Feature by Outcome")
plt.show()


In [ ]:

plt.figure(figsize=(8, 5))

groups = [
    group["f2"].dropna().values
    for _, group in df.groupby("y")
]

labels = [
    str(label)
    for label in sorted(df["y"].dropna().unique())
]

plt.boxplot(
    groups,
    tick_labels=labels
)

plt.xlabel("30-Day Mortality Outcome")
plt.ylabel("f2: Condition Hierarchy Feature")
plt.title("Condition Hierarchy Feature by Outcome")
plt.show()



## 10. Compare Study Arms if Multiple Arms Are Present

Most breakout groups may have only one study arm in their file. If multiple arms were combined before upload, this section will compare them automatically.


In [ ]:

if df["study_arm"].nunique() > 1:

    arm_labels = list(df["study_arm"].dropna().unique())

    arm_data = [
        df.loc[df["study_arm"] == arm, "f1"].dropna().values
        for arm in arm_labels
    ]

    plt.figure(figsize=(10, 5))
    plt.boxplot(
        arm_data,
        tick_labels=arm_labels
    )
    plt.xlabel("Study Arm")
    plt.ylabel("f1")
    plt.title("Drug Exposure Feature by Study Arm")
    plt.xticks(rotation=45)
    plt.show()

else:
    print(
        "Only one study arm is present. "
        "Run this section again after combining group exports if desired."
    )


## 11. Correlation Matrix

In [ ]:

correlation_matrix = df[["f1", "f2", "y"]].corr()

display(correlation_matrix)

fig, ax = plt.subplots(figsize=(6, 5))

image = ax.imshow(correlation_matrix)

ax.set_xticks(range(len(correlation_matrix.columns)))
ax.set_yticks(range(len(correlation_matrix.columns)))

ax.set_xticklabels(correlation_matrix.columns)
ax.set_yticklabels(correlation_matrix.columns)

for i in range(len(correlation_matrix.columns)):
    for j in range(len(correlation_matrix.columns)):
        ax.text(
            j,
            i,
            f"{correlation_matrix.iloc[i, j]:.2f}",
            ha="center",
            va="center"
        )

plt.colorbar(image)
plt.title("Correlation Matrix")
plt.show()



## 12. Prepare the Modeling Dataset

For the simple logistic regression model:

- `X` contains the engineered predictors `f1` and `f2`
- `y` contains the 30-day mortality outcome

Rows with missing model variables will be removed.


In [ ]:

model_df = df[
    ["person_id", "study_arm", "f1", "f2", "y"]
].dropna().copy()

X = model_df[["f1", "f2"]]
y = model_df["y"].astype(int)

print(f"Modeling rows: {len(model_df):,}")
print()
print("Outcome counts:")
print(y.value_counts())


## 13. Split the Data into Training and Test Sets

In [ ]:

if y.nunique() < 2:
    raise ValueError(
        "The dataset contains only one outcome class. "
        "Logistic regression requires both y=0 and y=1."
    )

class_counts = y.value_counts()

can_stratify = class_counts.min() >= 2

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=715,
    stratify=y if can_stratify else None
)

print(f"Training rows: {len(X_train):,}")
print(f"Test rows: {len(X_test):,}")

if not can_stratify:
    print(
        "Stratified splitting was not used because one outcome class "
        "had fewer than two observations."
    )



## 14. Fit Logistic Regression

The pipeline standardizes `f1` and `f2` before fitting the model.

Standardization is useful here because the two engineered features may be on very different numeric scales.


In [ ]:

model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic_regression", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)

print("Model fitted successfully.")


## 15. Evaluate the Model

In [ ]:

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, zero_division=0),
    "recall": recall_score(y_test, y_pred, zero_division=0),
    "f1_score": f1_score(y_test, y_pred, zero_division=0)
}

if y_test.nunique() == 2:
    metrics["roc_auc"] = roc_auc_score(y_test, y_prob)
else:
    metrics["roc_auc"] = np.nan

metrics_df = pd.DataFrame(
    metrics.items(),
    columns=["metric", "value"]
)

display(metrics_df)


In [ ]:

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred
)

plt.title("Confusion Matrix")
plt.show()


In [ ]:

if y_test.nunique() == 2:
    RocCurveDisplay.from_predictions(
        y_test,
        y_prob
    )
    plt.title("ROC Curve")
    plt.show()
else:
    print(
        "ROC curve was skipped because the test set contains only one outcome class."
    )



## 16. Examine the Regression Coefficients

Because the model standardized `f1` and `f2`, these coefficients describe the change in log odds associated with a one-standard-deviation increase in each feature, holding the other feature constant.


In [ ]:

logistic_model = model.named_steps["logistic_regression"]

coefficient_df = pd.DataFrame({
    "feature": ["f1", "f2"],
    "coefficient": logistic_model.coef_[0]
})

coefficient_df["odds_ratio"] = np.exp(
    coefficient_df["coefficient"]
)

display(coefficient_df)

print(
    "Intercept:",
    logistic_model.intercept_[0]
)


## 17. Score Every Patient

In [ ]:

model_df["predicted_probability"] = model.predict_proba(
    model_df[["f1", "f2"]]
)[:, 1]

display(
    model_df.sort_values(
        "predicted_probability",
        ascending=False
    ).head(20)
)


## 18. Visualize Predicted Risk

In [ ]:

plt.figure(figsize=(8, 5))

plt.hist(
    model_df["predicted_probability"],
    bins=20
)

plt.xlabel("Predicted Probability")
plt.ylabel("Patients")
plt.title("Distribution of Predicted 30-Day Mortality Probability")
plt.show()



## 19. Export the Scored Patient Dataset

This creates a new CSV containing each patient's engineered features, observed outcome, and model-predicted probability.


In [ ]:

output_filename = "scored_omop_features.csv"

model_df.to_csv(
    output_filename,
    index=False
)

print(f"Created: {output_filename}")


In [ ]:

from google.colab import files

files.download("scored_omop_features.csv")



# Group Discussion Questions

Use your SQL results and Python analysis to answer the following:

1. How many patients were included in your assigned study arm?
2. What does `f1` represent?
3. What does `f2` represent?
4. How did the recursive condition hierarchy change the information represented by `f2`?
5. Which feature showed greater variability?
6. Were there substantial outliers?
7. What proportion of patients had `y = 1`?
8. What signs did the fitted coefficients have?
9. How should a positive versus negative coefficient be interpreted?
10. Did either feature appear strongly associated with the outcome?
11. How well did the model perform on the test data?
12. Why does model performance alone not establish a causal effect of the intervention?
13. What additional OMOP feature would your group add?
14. Which parts of the workflow were better suited to SQL?
15. Which parts were easier to perform in Python?

## Suggested Group Deliverables

Submit:

- the SQL used to create the feature table
- the exported CSV
- this completed Colab notebook
- at least two visualizations
- the fitted logistic regression coefficients
- a short interpretation of the results
- one proposed additional feature
